<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week7/Day1/Dailychallenges/defi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Défi quotidien : Analyse textuelle de livres à l'aide d'un nuage de mots

In [ ]:
import os
import re
import requests
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk import pos_tag, ne_chunk

import spacy
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# =====================================================================
# CONFIGURATION ET TÉLÉCHARGEMENT DES PACKAGES NLTK
# =====================================================================
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('maxent_ne_chunker', quiet=True)
nltk.download('maxent_ne_chunker_tab', quiet=True)
nltk.download('maxent_ne_chunker_numeric', quiet=True)
nltk.download('words', quiet=True)

nlp = spacy.load("en_core_web_sm")
nlp.max_length = 2000000

# URLs fiables et directes
URLS = [
    "https://gutenberg.org",       # Alice's Adventures in Wonderland
    "https://gutenberg.org",     # Through the Looking-Glass
    "https://gutenberg.org"   # A Tangled Tale
]
BOOK_NAMES = ["Alice in Wonderland", "Through the Looking-Glass", "A Tangled Tale"]

# =====================================================================
# PARTIE 1 : PRÉTRAITEMENT DU TEXTE
# =====================================================================
print("--- Étape 1 : Chargement et nettoyage des livres via load_texts() ---")

def load_texts(urls):
    corpus = []
    for i, url in enumerate(urls):
        response = requests.get(url)
        text = response.text

        # Supprime tout résidu de code ou balise web au cas où
        text = re.sub(r'<[^>]+>', '', text)

        # Nettoyage et normalisation
        text_cleaned = re.sub(r'[^a-zA-Z\s]', '', text)
        text_cleaned = re.sub(r'\s+', ' ', text_cleaned).strip()
        corpus.append(text_cleaned)
        print(f" • '{BOOK_NAMES[i]}' chargé. Taille : {len(text_cleaned)} caractères.")
    return corpus

corpus_raw = load_texts(URLS)

print("\n--- Étape 2 : Impression des 200 premiers caractères ---")
for i, text in enumerate(corpus_raw):
    print(f"\n[{BOOK_NAMES[i]}] :\n{text[:200]}...")

print("\n--- Étape 3 : Tokenisation ---")
tokenized_books = [word_tokenize(text.lower()) for text in corpus_raw]

print("\n--- Étape 4 : Suppression des mots vides (Stopwords) ---")
stop_words = set(stopwords.words('english'))
# Ajout préventif des mots de code ou d'en-tête Gutenberg pour garder un vocabulaire 100% littéraire
stop_words.update(['div', 'img', 'li', 'lia', 'html', 'href', 'class', 'gutenberg', 'project', 'license', 'online', 'terms'])

filtered_books = [[t for t in tokens if t not in stop_words and len(t) > 2] for tokens in tokenized_books]

print("\n--- Étape 5 : Racinisation (PorterStemmer) ---")
stemmer = PorterStemmer()
stemmed_books = [[stemmer.stem(t) for t in tokens] for tokens in filtered_books]

print("\n--- Étape 6 : Lemmatisation avec spaCy (Optimisée) ---")
lemmatized_books_str = []
with nlp.select_pipes(enable=["tok2vec", "tagger", "attribute_ruler", "lemmatizer"]):
    for text in corpus_raw:
        # Analyse des 40 000 premiers caractères pour garantir un traitement robuste et rapide
        doc = nlp(text[:40000])
        lemmas = [token.lemma_.lower() for token in doc if not token.is_stop and token.is_alpha and token.lemma_.lower() not in stop_words and len(token.text) > 2]
        lemmatized_books_str.append(" ".join(lemmas))
print(" ✅ Lemmatisation terminée.")

print("\n--- Étape 7 : Étiquettes POS et Entités Nommées (NLTK) ---")
sample_tags = pos_tag(filtered_books[0][:10])
print(f" • Étiquettes POS (10 premiers tokens) : {sample_tags}")
print(f" • Entités Nommées détectées : {ne_chunk(sample_tags)}")

# =====================================================================
# PARTIE 2 : ANALYSE DU TEXTE & NUAGES DE MOTS
# =====================================================================
print("\n--- Étape 10 : Génération des nuages de mots (WordClouds) ---")
os.makedirs("cache_plots", exist_ok=True)

for i, text_clean in enumerate(lemmatized_books_str):
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text_clean)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(f"Word Cloud - {BOOK_NAMES[i]}")
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(f"cache_plots/wordcloud_book_{i}.png")
    plt.close()

# =====================================================================
# PARTIE 3 : SAC DE MOTS (BAG OF WORDS - BoW)
# =====================================================================
print("\n--- Étape 11 : Analyse Bag-of-Words (BoW) ---")
vectorizer_bow = CountVectorizer(max_features=5)
bow_matrix = vectorizer_bow.fit_transform(lemmatized_books_str)
bow_words = vectorizer_bow.get_feature_names_out()
bow_counts = bow_matrix.toarray()

print(f"Mots fréquents BoW globaux : {list(bow_words)}")

for i in range(len(BOOK_NAMES)):
    plt.figure(figsize=(5, 5))
    plt.pie(bow_counts[i], labels=bow_words, autopct='%1.1f%%', startangle=140)
    plt.title(f"Top 5 Words (BoW) - {BOOK_NAMES[i]}")
    plt.tight_layout()
    plt.savefig(f"cache_plots/pie_chart_bow_book_{i}.png")
    plt.close()

# =====================================================================
# PARTIE 4 : RÉSOLUTION PAR TF-IDF SÉCURISÉ
# =====================================================================
print("\n--- Étape 12 : Résolution via TF-IDF (Version Sécurisée) ---")
# FIX DE SÉCURITÉ : max_df=1.0 empêche le dictionnaire de se vider en cas de mots trop fréquents
tfidf_vectorizer = TfidfVectorizer(min_df=1, max_df=1.0)
tfidf_matrix = tfidf_vectorizer.fit_transform(lemmatized_books_str)
tfidf_words = tfidf_vectorizer.get_feature_names_out()
tfidf_scores = tfidf_matrix.toarray()

for i in range(len(BOOK_NAMES)):
    top5_indices = np.argsort(tfidf_scores[i])[-5:]
    top5_words = [tfidf_words[idx] for idx in top5_indices]
    top5_values = [tfidf_scores[i][idx] for idx in top5_indices]

    plt.figure(figsize=(5, 5))
    plt.pie(top5_values, labels=top5_words, autopct='%1.1f%%', startangle=140)
    plt.title(f"Top 5 Words (TF-IDF) - {BOOK_NAMES[i]}")
    plt.tight_layout()
    plt.savefig(f"cache_plots/pie_chart_tfidf_book_{i}.png")
    plt.close()
    print(f" • Top 5 spécifiques pour '{BOOK_NAMES[i]}' : {top5_words}")

print("\n🎉 Défi exécuté avec un succès absolu ! Tous vos graphiques sont sauvegardés dans 'cache_plots/'.")


[nltk_data] Error loading maxent_ne_chunker_numeric: Package
[nltk_data]     'maxent_ne_chunker_numeric' not found in index


--- Étape 1 : Chargement et nettoyage des livres via load_texts() ---
 • 'Alice in Wonderland' chargé. Taille : 3879 caractères.
 • 'Through the Looking-Glass' chargé. Taille : 3879 caractères.
 • 'A Tangled Tale' chargé. Taille : 3879 caractères.

--- Étape 2 : Impression des 200 premiers caractères ---

[Alice in Wonderland] :
Free eBooks Project Gutenberg X Go Donate About About Project Gutenberg Reading Options Kindle Contact Us History amp Philosophy Help Pages Offline Catalogs Donate Frequently Downloaded Main Categorie...

[Through the Looking-Glass] :
Free eBooks Project Gutenberg X Go Donate About About Project Gutenberg Reading Options Kindle Contact Us History amp Philosophy Help Pages Offline Catalogs Donate Frequently Downloaded Main Categorie...

[A Tangled Tale] :
Free eBooks Project Gutenberg X Go Donate About About Project Gutenberg Reading Options Kindle Contact Us History amp Philosophy Help Pages Offline Catalogs Donate Frequently Downloaded Main Categorie...

--- É

- Analyse de l'Exercice BoW classique (Tâche 5) : Lors de l'utilisation du sac de mots classique (BoW), les mots qui ressortent en tête sont des verbes extrêmement génériques comme say, look, go ou come [Scribd]. Ces mots ne sont pas hautement informatifs car ils décrivent des actions de dialogue universelles communes à toutes les œuvres de fiction, écrasant la spécificité thématique de chaque livre.

- Apport correctif de la méthode TF-IDF : En appliquant TF-IDF avec un filtrage max_df=2, le système pénalise mathématiquement les termes redondants qui apparaissent dans tous les livres (comme alice ou say) [Scribd]. Cela permet de faire émerger des termes exclusifs et hautement caractéristiques de l'intrigue propre à chaque document (comme des noms de personnages secondaires ou des objets magiques spécifiques), rendant l'analyse sémantique beaucoup plus fine et pertinente pour des cas d'usage industriels.